# 45 — Identity probe

Decision D2 rests on a claim about the published standard: **a COSMoS instance can
be schema-valid or RDF-convertible, not both.** This notebook measures that claim
against real rows from the pinned export, and asserts the outcome of every step —
so if CDISC changes the schema, this notebook **fails**, and the failure is the
news.

It is not part of the build. Nothing downstream consumes its output; it exists to
keep the argument honest and reproducible.

Five attempts, two real concepts:

| # | Schema | Instance | Expected |
|---|---|---|---|
| 1 | as published | bare C-code | conversion fails — no prefix to expand a bare code |
| 2 | `+ id_prefixes: [NCIT]` | bare C-code | still fails — `id_prefixes` constrains, it does not expand |
| 3 | as published | CURIE-lifted | **validation** fails — the pattern forbids the convertible form |
| 4 | `+ relaxed pattern`, `range: uriorcurie` | CURIE-lifted | converts; subject is the OBO PURL |
| 5 | same as 4 | `NEW_` placeholder | fails — no prefix exists, and none can |

Attempt 5 is the point. Attempts 1–4 describe a fixable inconvenience; attempt 5
is a concept the standard cannot name.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
BUILD     = "../build"
REPORTS   = "../reports"

BC_EXPORT  = f"{DOWNLOADS}/cdisc_biomedical_concepts_latest.csv"
DSS_EXPORT = f"{DOWNLOADS}/cdisc_sdtm_dataset_specializations_latest.csv"
BC_MODEL   = f"{BUILD}/cosmos_bc_model.patched.yaml"

# Two real concepts from the pinned export. Nothing here is invented.
IDENTIFIED_BC   = "C115805"    # 6MWT - Distance at 6 Minutes; carries a LOINC coding
UNIDENTIFIED_BC = "NEW_LZZT1"  # TTS Acceptability Survey - Patch Appearance

NCIT_PURL = "http://purl.obolibrary.org/obo/NCIT_"

## Build the two instances from the pinned CSV

Assembled programmatically from real rows — no value is typed by hand. The flat
export repeats a concept's `coding` on every DEC row, so codings are deduplicated;
that is a flattening artifact, not data.

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

Path(BUILD).mkdir(parents=True, exist_ok=True)
Path(REPORTS).mkdir(parents=True, exist_ok=True)

bc_export = pd.read_csv(BC_EXPORT, dtype=str, keep_default_na=False)


def split(value):
    return [part for part in value.split(";") if part]


def build_instance(bc_id):
    rows = bc_export[bc_export.bc_id == bc_id]
    if rows.empty:
        raise RuntimeError(f"{bc_id} not present in the pinned export")
    rows = rows[rows.package_date == rows.package_date.max()]
    head = rows.iloc[0]

    instance = {
        "packageDate": head.package_date,
        "packageType": "bc",
        "conceptId": head.bc_id,
        "shortName": head.short_name,
        "definition": head.definition,
        "categories": split(head.bc_categories),
    }
    if head.ncit_code:
        instance["ncitCode"] = head.ncit_code
    if head.parent_bc_id:
        instance["parentConceptId"] = head.parent_bc_id
    if split(head.synonyms):
        instance["synonyms"] = split(head.synonyms)
    if split(head.result_scales):
        instance["resultScales"] = split(head.result_scales)

    codings, seen = [], set()
    for _, row in rows.iterrows():
        if row.code and (row.code, row.system) not in seen:
            seen.add((row.code, row.system))
            codings.append({"code": row.code, "system": row.system, "systemName": row.system_name})
    if codings:
        instance["coding"] = codings

    decs = []
    for _, row in rows.drop_duplicates("dec_id").iterrows():
        if not row.dec_id:
            continue
        dec = {"conceptId": row.dec_id, "shortName": row.dec_label, "dataType": row.data_type}
        if row.ncit_dec_code:
            dec["ncitCode"] = row.ncit_dec_code
        decs.append(dec)
    if decs:
        instance["dataElementConcepts"] = decs

    return instance


instances = {}
for label, bc_id in (("identified", IDENTIFIED_BC), ("unidentified", UNIDENTIFIED_BC)):
    instance = build_instance(bc_id)
    path = Path(BUILD, f"probe_{label}.yaml")
    path.write_text(yaml.safe_dump(instance, sort_keys=False, allow_unicode=True), encoding="utf-8")
    instances[label] = path
    print(f"{label:13s} {bc_id:12s} {instance['shortName']}")

## CURIE-lifted copies

The same instances with every C-code rewritten as an `NCIT:` CURIE. This is the
only form a JSON-LD context or an RDF dumper can turn into a subject IRI. A
`NEW_` value is left alone — there is no prefix to lift it into, which is the
finding.

In [ ]:
import re

CCODE = re.compile(r"^C[0-9]+$")


def lift(value):
    return f"NCIT:{value}" if CCODE.match(value) else value


for label, path in list(instances.items()):
    instance = yaml.safe_load(path.read_text(encoding="utf-8"))
    instance["conceptId"] = lift(instance["conceptId"])
    if "parentConceptId" in instance:
        instance["parentConceptId"] = lift(instance["parentConceptId"])
    for dec in instance.get("dataElementConcepts", []):
        dec["conceptId"] = lift(dec["conceptId"])

    lifted = Path(BUILD, f"probe_{label}_curie.yaml")
    lifted.write_text(yaml.safe_dump(instance, sort_keys=False, allow_unicode=True), encoding="utf-8")
    instances[f"{label}_curie"] = lifted
    print(f"{lifted.name:32s} conceptId = {instance['conceptId']}")

## Schema variants

Three, all derived from the patched published model by explicit substitution. Each
substitution is asserted to match exactly once, so an upstream change breaks the
notebook instead of quietly changing what it proves.

In [ ]:
published = Path(BC_MODEL).read_text(encoding="utf-8")

PUBLISHED_CONCEPT_ID = """  conceptId:
    description: An identifier that uniquely represents an entity
    identifier: true
    range: string
    pattern: "^(C[0-9]+|NEW_[A-Z_]*[0-9]*)$"
    required: true"""

RELAXED_CONCEPT_ID = """  conceptId:
    description: An identifier that uniquely represents an entity
    identifier: true
    range: uriorcurie
    pattern: "^(NCIT:C[0-9]+|NEW_[A-Z_]*[0-9]*)$"
    required: true"""

if published.count(PUBLISHED_CONCEPT_ID) != 1:
    raise RuntimeError("the published conceptId slot no longer has the expected shape")

with_id_prefixes = published.replace(
    "  BiomedicalConcept:\n    tree_root: true\n",
    "  BiomedicalConcept:\n    tree_root: true\n    id_prefixes:\n      - NCIT\n",
)
if with_id_prefixes == published:
    raise RuntimeError("could not attach id_prefixes to BiomedicalConcept")

relaxed = published.replace(PUBLISHED_CONCEPT_ID, RELAXED_CONCEPT_ID)

schemas = {}
for label, text in (
    ("published", published),
    ("id_prefixes", with_id_prefixes),
    ("relaxed", relaxed),
):
    path = Path(BUILD, f"probe_schema_{label}.yaml")
    path.write_text(text, encoding="utf-8")
    schemas[label] = path
    print(f"{label:12s} {path}")

## Run the five attempts

`linkml-validate` and `linkml-convert` are invoked as the console scripts sitting
beside the running interpreter, so the probe uses whatever environment the
notebook itself runs in.

In [ ]:
import subprocess
import sys

BIN = Path(sys.executable).parent


def run(tool, schema, instance, *extra):
    result = subprocess.run(
        [str(BIN / tool), "--schema", str(schema), "--target-class", "BiomedicalConcept",
         *extra, str(instance)],
        capture_output=True,
        text=True,
    )
    return result


def last_line(text):
    lines = [line for line in text.strip().splitlines() if line.strip()]
    return lines[-1] if lines else ""


ATTEMPTS = [
    ("1", "convert", "published",   "identified",        "FAIL",    "Unknown CURIE prefix: @base"),
    ("2", "convert", "id_prefixes", "identified",        "FAIL",    "Unknown CURIE prefix: @base"),
    ("3", "validate", "published",  "identified_curie",  "FAIL",    "does not match"),
    ("4", "convert", "relaxed",     "identified_curie",  "SUCCESS", NCIT_PURL),
    ("5", "convert", "relaxed",     "unidentified_curie", "FAIL",   "Unknown CURIE prefix: @base"),
]

rows, failures = [], []
converted = {}

for number, tool, schema, instance, expectation, signal in ATTEMPTS:
    extra = ("-t", "ttl") if tool == "convert" else ()
    result = run(f"linkml-{tool}", schemas[schema], instances[instance], *extra)

    if tool == "validate":
        outcome = "SUCCESS" if "No issues found" in result.stdout else "FAIL"
        detail = last_line(result.stdout) or last_line(result.stderr)
    else:
        outcome = "SUCCESS" if result.returncode == 0 else "FAIL"
        detail = last_line(result.stderr) if outcome == "FAIL" else "converted"
        if outcome == "SUCCESS":
            converted[number] = result.stdout

    matched = signal in (detail + result.stdout)
    ok = outcome == expectation and (matched or outcome == "SUCCESS")
    if not ok:
        failures.append(number)

    rows.append({
        "attempt": number,
        "tool": f"linkml-{tool}",
        "schema": schema,
        "instance": instances[instance].name,
        "expected": expectation,
        "outcome": outcome,
        "detail": detail[:300],
    })
    print(f"{number}  {expectation:8s} -> {outcome:8s}  {detail[:110]}")

## The subject IRI attempt 4 produces

Read from the converted graph rather than from the Turtle text, so the prefix
form cannot flatter the result.

In [ ]:
from rdflib import Graph, URIRef
from rdflib.namespace import RDF

graph = Graph().parse(data=converted["4"], format="turtle")
expected_subject = URIRef(NCIT_PURL + IDENTIFIED_BC)

subjects = sorted(str(s) for s in graph.subjects(RDF.type, None) if isinstance(s, URIRef))
if str(expected_subject) not in subjects:
    failures.append("4-subject")
    print(f"FAIL  expected subject {expected_subject} not in the graph")
else:
    print(f"ok    subject IRI  {expected_subject}")

print(f"      typed subjects {len(subjects)} (1 BiomedicalConcept + its DECs)")
print(f"      blank nodes    {len({s for s in graph.subjects() if not isinstance(s, URIRef)})} (the LOINC coding has no IRI)")

## The concepts that cannot be identified

Decision D2 renders no node for these — the repo invents nothing. Recording them
here is what keeps that absence legible: a consumer can tell a deliberate gap from
an oversight, and the list is derived from the pinned export rather than asserted.

In [ ]:
import csv

dss_export = pd.read_csv(DSS_EXPORT, dtype=str, keep_default_na=False)

unidentified = []
for column, label_column in (("bc_id", "short_name"), ("dec_id", "dec_label")):
    subset = bc_export[bc_export[column].str.startswith("NEW_")]
    for value in sorted(subset[column].unique()):
        rows_for = subset[subset[column] == value]
        head = rows_for.iloc[0]
        unidentified.append({
            "layer": "BC export",
            "column": column,
            "value": value,
            "label": head[label_column],
            "first_package_date": rows_for.package_date.min(),
            "referenced_by_dss": bool((dss_export[column] == value).any()) if column in dss_export else False,
        })

report = Path(REPORTS, "unidentified_concepts.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(unidentified[0]))
    writer.writeheader()
    writer.writerows(unidentified)

print(f"wrote {report}: {len(unidentified)} concepts with no NCIt code")
for row in unidentified:
    print(f"   {row['column']:8s} {row['value']:12s} {row['label'][:52]:54s} dss={row['referenced_by_dss']}")

identified = bc_export[bc_export.ncit_code != ""].bc_id.nunique()
total = bc_export.bc_id.nunique()
print()
print(f"{identified:,} of {total:,} biomedical concepts carry an NCIt code; {total - identified} do not")

## Result

In [ ]:
report = Path(REPORTS, "identity_probe.csv")
with open(report, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"wrote {report}")

if failures:
    raise RuntimeError(
        f"attempt(s) {', '.join(failures)} did not behave as decision D2 records. "
        "Either the toolchain or the published schema has changed - read the row detail "
        "before editing anything, because this notebook failing may be good news."
    )
print("\nAll five attempts behaved as D2 records.")